In [12]:
import openseespywin as ops
import opsvis as opsv 
import numpy as np
import ipywidgets as widgets
import os
import matplotlib.pyplot as plt
import math
import opstool as opst
#import eurocodepy as ecpy
import time as tt
import gmsh
# import openseespy.postprocessing.Get_Rendering as opsplt

In [13]:
import sys
sys.path.append('Data')
import functions.solver_function as SF
import importlib
importlib.reload(SF)
from functions.solver_function import *

In [14]:
opst.add_ops_hints_file()

OPSTOOL :: opensees.pyi file has been created to 
C:\Users\micha\miniconda3\envs\blue_env\Lib\site-packages\openseespywin\opensees.pyi!

In [15]:
length_unit = "m"  # base unit
force_unit = "KN"  # base unit

UNIT = opst.pre.UnitSystem(length=length_unit, force=force_unit)

print("Length:", UNIT.mm, UNIT.mm2, UNIT.cm, UNIT.m,)
print("Force", UNIT.N, UNIT.kN, )
print("Stress", UNIT.MPa, UNIT.kPa, UNIT.Pa, )
print("Mass", UNIT.g, UNIT.kg, UNIT.ton)
print(UNIT)

Length: 0.001 1e-06 0.01 1
Force 0.001 1
Stress 1000.0 1.0 0.001
Mass 1e-06 0.001 1.0
<UnitSystem: length='m', force='kn', time='sec' (145363957440)>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import os

# Load data
a1 = np.loadtxt('acc')
d1 = np.loadtxt('disp')
p1 = np.loadtxt('pwp')
s1 = np.loadtxt('stress1')
e1 = np.loadtxt('strain1')
s5 = np.loadtxt('stress5')
e5 = np.loadtxt('strain5')
s9 = np.loadtxt('stress9')
e9 = np.loadtxt('strain9')

fs = [0.5, 0.2, 4, 6]  # paper position (unused in Python)
accMul = 2

def compute_p_q(stress):
    po = np.mean(stress[:, 1:4], axis=1)
    qo = []
    for i in range(len(stress)):
        q = ((stress[i,1] - stress[i,2])**2 +
             (stress[i,2] - stress[i,3])**2 +
             (stress[i,1] - stress[i,3])**2 +
             6.0 * stress[i,4]**2)
        q = np.sign(stress[i,4]) * (1/3.0) * np.sqrt(q)
        qo.append(q)
    return po, np.array(qo)

# --- Integration Point 1 ---
po, qo = compute_p_q(s1)
plt.figure(1)
plt.subplot(2,1,1)
plt.plot(e1[:,3], s1[:,4], 'r')
plt.title('Shear stress τₓᵧ VS. Shear strain εₓᵧ at 10 m depth')
plt.xlabel('Shear strain εₓᵧ')
plt.ylabel('Shear stress τₓᵧ (kPa)')

plt.subplot(2,1,2)
plt.plot(-po, qo, 'r')
plt.title('Confinement p VS. Deviatoric stress q at 10 m depth')
plt.xlabel('Confinement p (kPa)')
plt.ylabel('q (kPa)')
plt.tight_layout()
plt.savefig('SS_PQ_10m.jpg')
plt.close()

# --- Integration Point 5 ---
po, qo = compute_p_q(s5)
plt.figure(5)
plt.subplot(2,1,1)
plt.plot(e5[:,3], s5[:,4], 'r')
plt.title('Shear stress τₓᵧ VS. Shear strain εₓᵧ at 6 m depth')
plt.xlabel('Shear strain εₓᵧ')
plt.ylabel('Shear stress τₓᵧ (kPa)')

plt.subplot(2,1,2)
plt.plot(-po, qo, 'r')
plt.title('Confinement p VS. Deviatoric stress q at 6 m depth')
plt.xlabel('Confinement p (kPa)')
plt.ylabel('q (kPa)')
plt.tight_layout()
plt.savefig('SS_PQ_6m.jpg')
plt.close()

# --- Integration Point 9 ---
po, qo = compute_p_q(s9)
plt.figure(6)
plt.subplot(2,1,1)
plt.plot(e9[:,3], s9[:,4], 'r')
plt.title('Shear stress τₓᵧ VS. Shear strain εₓᵧ at 2 m depth')
plt.xlabel('Shear strain εₓᵧ')
plt.ylabel('Shear stress τₓᵧ (kPa)')

plt.subplot(2,1,2)
plt.plot(-po, qo, 'r')
plt.title('Confinement p VS. Deviatoric stress q at 2 m depth')
plt.xlabel('Confinement p (kPa)')
plt.ylabel('q (kPa)')
plt.tight_layout()
plt.savefig('SS_PQ_2m.jpg')
plt.close()

# --- Displacement wrt base ---
plt.figure(2)
plt.subplot(2,1,1)
a, = plt.plot(d1[:,0], d1[:,7], 'r')
b, = plt.plot(d1[:,0], d1[:,13], 'g')
c, = plt.plot(d1[:,0], d1[:,21], 'b')
plt.title('Lateral displacement wrt base')
plt.xlabel('Time (s)')
plt.ylabel('Displacement (m)')
plt.legend([a,b,c], ['8m depth', '4m depth', 'Surface'], loc='upper right')
plt.tight_layout()
plt.savefig('Disp.jpg')
plt.close()

# --- Acceleration computation ---
t_interp = np.arange(0, 20 * np.pi, np.pi / 50)
s = accMul * np.sin(t_interp)
s = np.concatenate([s, np.zeros(3000)])
interp_func = interp1d(np.arange(0, 0.01 * len(s), 0.01), s, fill_value="extrapolate")
s1_interp = interp_func(a1[:,0])

plt.figure(3)
plt.subplot(3,1,1)
a, = plt.plot(a1[:,0], s1_interp + a1[:,21], 'r')
plt.legend([a], ['at surface'], loc='lower right')
plt.title('Lateral acceleration')
plt.xlabel('Time (s)')
plt.ylabel('Acceleration (m/s²)')

plt.subplot(3,1,2)
a, = plt.plot(a1[:,0], s1_interp + a1[:,13], 'r')
plt.legend([a], ['4 m depth'], loc='lower right')
plt.xlabel('Time (s)')
plt.ylabel('Acceleration (m/s²)')

plt.subplot(3,1,3)
a, = plt.plot(a1[:,0], s1_interp + a1[:,7], 'r')
plt.legend([a], ['8 m depth'], loc='lower right')
plt.xlabel('Time (s)')
plt.ylabel('Acceleration (m/s²)')

plt.tight_layout()
plt.savefig('Acc.jpg')
plt.close()

# --- Pore Pressure ---
plt.figure(4)
plt.subplot(3,1,1)
a, = plt.plot(p1[:,0], p1[:,10], 'r')
plt.legend([a], ['1 m depth'], loc='lower right')
plt.title('Pore pressure')
plt.xlabel('Time (s)')
plt.ylabel('Pore pressure (kPa)')

plt.subplot(3,1,2)
a, = plt.plot(p1[:,0], p1[:,6], 'r')
plt.legend([a], ['5 m depth'], loc='lower right')
plt.xlabel('Time (s)')
plt.ylabel('Pressure (kPa)')

plt.subplot(3,1,3)
a, = plt.plot(p1[:,0], p1[:,1], 'r')
plt.legend([a], ['10 m depth'], loc='lower right')
plt.xlabel('Time (s)')
plt.ylabel('Pressure (kPa)')

plt.tight_layout()
plt.savefig('EPWP.jpg')
plt.close()
